## Step 1 — Install packages

In [5]:
import subprocess, sys
pkgs = ['lightgbm', 'pyarrow', 'pandas', 'numpy', 'scikit-learn', 'joblib']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
print('All packages ready.')

All packages ready.


## Step 2 — Imports

In [6]:
import os, gc, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
import joblib
from sklearn.metrics import classification_report, accuracy_score
warnings.filterwarnings('ignore')
print('Imports OK.')

Imports OK.


## Step 3 — Set paths  
**Edit only these two lines to point to your files.**

In [8]:
# ============================================================
# CELL 3 - Set paths for FULL DATASET (Kaggle or Local Fallback)
# ============================================================

import os
from pathlib import Path

TRAIN_PATH = None
TEST_PATH = None

candidate_dirs = ['/kaggle/input', '../../Dataset/EMBER', 'Dataset/EMBER']
for cdir in candidate_dirs:
    if os.path.exists(cdir):
        for root, dirs, files in os.walk(cdir):
            for file in files:
                if file == 'train_ember_2018_v2_features.parquet':
                    TRAIN_PATH = os.path.join(root, file)
                elif file == 'test_ember_2018_v2_features.parquet':
                    TEST_PATH = os.path.join(root, file)
    if TRAIN_PATH and TEST_PATH:
        break

if TRAIN_PATH is None or TEST_PATH is None:
    print("❌ ERROR: Could not find parquet files in candidate directories!")
    raise RuntimeError("Parquet dataset files not found.")

if os.path.exists('/kaggle/working'):
    MODEL_OUT = '/kaggle/working/aegis_ember_model_full.pkl'
else:
    MODEL_OUT = '../../trained_models/ember/aegis_ember_model_full.pkl'

TRAIN_SAMPLE = None   # None = load ALL rows
TEST_SAMPLE = None    # None = load ALL rows

print(f"📊 Training on: ALL available rows")
print(f"✓ Train path: {TRAIN_PATH}")
print(f"✓ Test path: {TEST_PATH}")
print(f"✓ Model output: {MODEL_OUT}")


✅ Found test file at: /kaggle/input/datasets/dikshaparulekar/ember-2018/test_ember_2018_v2_features.parquet
✅ Found train file at: /kaggle/input/datasets/dikshaparulekar/ember-2018/train_ember_2018_v2_features.parquet

📊 Training on: ALL available rows (full dataset)
📊 Testing on: ALL available rows
✓ Train path exists: True
✓ Test path exists: True
✓ Model will be saved to: /kaggle/working/aegis_ember_model_full.pkl


In [11]:
# ============================================================
# CELL 3 - Set paths for FULL DATASET (Kaggle or Local Fallback)
# ============================================================

import os
from pathlib import Path

TRAIN_PATH = None
TEST_PATH = None

candidate_dirs = ['/kaggle/input', '../../Dataset/EMBER', 'Dataset/EMBER']
for cdir in candidate_dirs:
    if os.path.exists(cdir):
        for root, dirs, files in os.walk(cdir):
            for file in files:
                if file == 'train_ember_2018_v2_features.parquet':
                    TRAIN_PATH = os.path.join(root, file)
                elif file == 'test_ember_2018_v2_features.parquet':
                    TEST_PATH = os.path.join(root, file)
    if TRAIN_PATH and TEST_PATH:
        break

if TRAIN_PATH is None or TEST_PATH is None:
    print("❌ ERROR: Could not find parquet files in candidate directories!")
    raise RuntimeError("Parquet dataset files not found.")

if os.path.exists('/kaggle/working'):
    MODEL_OUT = '/kaggle/working/aegis_ember_model_full.pkl'
else:
    MODEL_OUT = '../../trained_models/ember/aegis_ember_model_full.pkl'

TRAIN_SAMPLE = None   # None = load ALL rows
TEST_SAMPLE = None    # None = load ALL rows

print(f"📊 Training on: ALL available rows")
print(f"✓ Train path: {TRAIN_PATH}")
print(f"✓ Test path: {TEST_PATH}")
print(f"✓ Model output: {MODEL_OUT}")


All items in /kaggle/input:
  'datasets'
    Files: ['dikshaparulekar']...


## Step 4 — Load data (memory-safe streaming)

In [9]:
# ============================================================
# CELL 4 - Load data (works with FULL dataset or samples)
# ============================================================

def stream_sample(path, n, seed=42):
    """Read rows. If n is None, loads ALL rows."""
    rng = np.random.default_rng(seed)
    meta = pq.read_metadata(path)
    pf = pq.ParquetFile(path)
    
    total_rows = meta.num_rows
    if n is None:
        print(f'  Total rows in file: {total_rows:,}  |  Loading ALL rows')
        n = total_rows  # Set to total rows
    else:
        print(f'  Total rows in file: {total_rows:,}  |  Taking {n:,} rows')
    
    row_groups = meta.num_row_groups
    
    # Randomize row group order for sampling
    order = rng.permutation(row_groups).tolist()
    parts, seen = [], 0
    
    for i in order:
        chunk = pf.read_row_group(i).to_pandas()
        parts.append(chunk)
        seen += len(chunk)
        if seen >= n:
            break
    
    df = pd.concat(parts, ignore_index=True)
    if len(df) > n:
        df = df.sample(n=n, random_state=seed).reset_index(drop=True)
    
    # Normalise label column regardless of capitalisation
    lc = next((c for c in df.columns if c.lower() == 'label'), None)
    if lc is None:
        raise RuntimeError(f'No label column found. Columns: {df.columns.tolist()[:15]}')
    df = df.rename(columns={lc: 'label'})
    return df

print('Loading training data...')
train_df = stream_sample(TRAIN_PATH, TRAIN_SAMPLE)
print(f'  Loaded {len(train_df):,} rows')

print('\nLoading test data...')
test_df = stream_sample(TEST_PATH, TEST_SAMPLE)
print(f'  Loaded {len(test_df):,} rows')

# Drop unlabelled rows (-1 = unknown in EMBER)
train_df = train_df[train_df['label'] != -1].reset_index(drop=True)
test_df = test_df[test_df['label'] != -1].reset_index(drop=True)

print(f'\nTrain: {train_df.shape} | label counts: {train_df["label"].value_counts().to_dict()}')
print(f'Test: {test_df.shape} | label counts: {test_df["label"].value_counts().to_dict()}')

Loading training data...
  Total rows in file: 799,912  |  Loading ALL rows
  Loaded 799,912 rows

Loading test data...
  Total rows in file: 199,956  |  Loading ALL rows
  Loaded 199,956 rows

Train: (599920, 2382) | label counts: {0.0: 299991, 1.0: 299929}
Test: (199956, 2382) | label counts: {0.0: 99985, 1.0: 99971}


## Step 5 — Prepare features

In [10]:
DROP = ['label', 'sha256']

X_train = train_df.drop(columns=DROP, errors='ignore').select_dtypes(include=[np.number])
y_train = train_df['label'].astype(int)

X_test  = test_df.drop(columns=DROP, errors='ignore').select_dtypes(include=[np.number])
y_test  = test_df['label'].astype(int)

# Make sure test has exactly the same columns as train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Clean up infinities and NaNs
for df in [X_train, X_test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)

# Free raw data frames to save RAM
del train_df, test_df
gc.collect()

print(f'Features : {X_train.shape[1]}')
print(f'Train rows: {len(X_train):,}  |  Test rows: {len(X_test):,}')
print(f'Train class balance — Benign: {(y_train==0).sum():,}  Malicious: {(y_train==1).sum():,}')

Features : 2381
Train rows: 599,920  |  Test rows: 199,956
Train class balance — Benign: 299,991  Malicious: 299,929


## Step 6 — Train LightGBM

In [11]:
# ============================================================
# CELL 6 - Train LightGBM with Validation Split (No Test Leakage)
# ============================================================
from sklearn.model_selection import train_test_split

# Create a clean validation split from X_train to prevent test set leakage
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.10, random_state=42, stratify=y_train
)

model = lgb.LGBMClassifier(
    n_estimators  = 500,
    learning_rate = 0.05,
    max_depth     = 12,
    num_leaves    = 31,
    objective     = 'binary',
    class_weight  = 'balanced',
    n_jobs        = -1,
    random_state  = 42,
    verbose       = -1,
)

print('Training started...')
model.fit(
    X_tr, y_tr,
    eval_set  = [(X_val, y_val)],
    callbacks = [
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ],
)
print('Training complete.')


Training started...
[100]	valid_0's binary_logloss: 0.215347
[200]	valid_0's binary_logloss: 0.165301
[300]	valid_0's binary_logloss: 0.143864
[400]	valid_0's binary_logloss: 0.131672
[500]	valid_0's binary_logloss: 0.122643
Training complete.


## Step 7 — Evaluate

In [12]:
# ============================================================
# CELL 7 - Evaluate
# ============================================================

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print('=' * 50)
print(f'First 10 threat scores: {y_proba[:10].round(3)}')

              precision    recall  f1-score   support

      Benign       0.96      0.95      0.96     99985
   Malicious       0.95      0.96      0.96     99971

    accuracy                           0.96    199956
   macro avg       0.96      0.96      0.96    199956
weighted avg       0.96      0.96      0.96    199956

Accuracy : 0.9567
First 10 threat scores: [0.999 0.034 0.998 0.039 0.001 0.998 0.004 0.004 0.962 0.094]


## Step 8 — Save model

In [14]:
os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)

joblib.dump({
    'model'       : model,
    'features'    : X_train.columns.tolist(),
    'classes'     : [0, 1],
    'class_names' : ['benign', 'malicious'],
    'version'     : '1.0-EMBER',
    'metadata'    : 'AEGIS Layer 1 — Supervised File Classifier',
}, MODEL_OUT)

size_mb = os.path.getsize(MODEL_OUT) / 1e6
print(f'Saved: {MODEL_OUT}')
print(f'Size : {size_mb:.1f} MB')


Saved: /kaggle/working/aegis_ember_model_full.pkl
Size : 1.9 MB
Done! Download from the Output tab on the right sidebar.


In [15]:
# ============================================================
# VERIFY MODEL TRAINING QUALITY
# ============================================================

import numpy as np
import pandas as pd

# Check if model actually learned something
print("=" * 50)
print("MODEL VERIFICATION")
print("=" * 50)

# 1. Check if model has trained trees
n_trees = model.n_estimators_
print(f"\n1. Number of trees trained: {n_trees}")

# 2. Check feature importance (should NOT be all zeros)
importance = model.feature_importances_
print(f"2. Feature importance range: {importance.min():.2e} to {importance.max():.2f}")
print(f"   Non-zero features: {(importance > 0).sum():,} / {len(importance):,}")

# 3. Check predictions (should NOT be constant)
y_pred_proba = model.predict_proba(X_test)[:, 1]
print(f"\n3. Prediction probabilities:")
print(f"   Min: {y_pred_proba.min():.4f}")
print(f"   Max: {y_pred_proba.max():.4f}")
print(f"   Mean: {y_pred_proba.mean():.4f}")
print(f"   Std: {y_pred_proba.std():.4f}")

# 4. Check if model is better than random guessing (0.5 accuracy)
from sklearn.metrics import accuracy_score
y_pred = (y_pred_proba > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n4. Accuracy: {accuracy:.4f}")
print(f"   Random guess would be: 0.5000")
if accuracy > 0.6:
    print(f"   ✅ Model IS learning (better than random)")
else:
    print(f"   ❌ Model NOT learning (same as random)")

# 5. Check class balance in predictions
print(f"\n5. Predicted class distribution:")
print(f"   Predicted Benign: {(y_pred == 0).sum():,}")
print(f"   Predicted Malicious: {(y_pred == 1).sum():,}")
print(f"   Actual Benign: {(y_test == 0).sum():,}")
print(f"   Actual Malicious: {(y_test == 1).sum():,}")

print("\n" + "=" * 50)
if accuracy > 0.85:
    print("✅ MODEL IS PROPERLY TRAINED! Good results.")
elif accuracy > 0.6:
    print("⚠️ Model learned something but could be better.")
else:
    print("❌ Model failed to learn - check data.")

MODEL VERIFICATION

1. Number of trees trained: 500
2. Feature importance range: 0.00e+00 to 325.00
   Non-zero features: 1,464 / 2,381

3. Prediction probabilities:
   Min: 0.0000
   Max: 0.9999
   Mean: 0.5150
   Std: 0.4475

4. Accuracy: 0.9567
   Random guess would be: 0.5000
   ✅ Model IS learning (better than random)

5. Predicted class distribution:
   Predicted Benign: 98,648
   Predicted Malicious: 101,308
   Actual Benign: 99,985
   Actual Malicious: 99,971

✅ MODEL IS PROPERLY TRAINED! Good results.


In [16]:
# ============================================================
# COMPARE TRAINING vs TEST ACCURACY
# ============================================================

from sklearn.metrics import accuracy_score

# Get predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Calculate accuracies
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("=" * 50)
print("📊 TRAINING vs TEST ACCURACY")
print("=" * 50)
print(f"\nTraining Accuracy:  {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test Accuracy:      {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Difference:         {train_acc - test_acc:.4f} ({((train_acc - test_acc)*100):.2f}%)")

# Check for overfitting
print("\n" + "=" * 50)
print("📈 OVERFITTING CHECK")
print("=" * 50)

if train_acc - test_acc < 0.02:
    print("✅ VERY GOOD: Small gap - model generalizes well!")
elif train_acc - test_acc < 0.05:
    print("⚠️ OKAY: Moderate gap - slight overfitting")
else:
    print("❌ POOR: Large gap - model is overfitting")

print(f"\nTraining rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

📊 TRAINING vs TEST ACCURACY

Training Accuracy:  0.9707 (97.07%)
Test Accuracy:      0.9567 (95.67%)
Difference:         0.0141 (1.41%)

📈 OVERFITTING CHECK
✅ VERY GOOD: Small gap - model generalizes well!

Training rows: 599,920
Test rows: 199,956
